# Phase 1: Label Construction for Predictive Failure Modeling

**Objective:** Build ground truth labels based on **GADS forced outages** to predict unplanned failures, excluding routine planned maintenance.

## Why GADS-Based Labels?
- **Planned outages** (PO) should NOT be predicted - they're scheduled maintenance
- **Reserve shutdowns** (RS) should NOT be predicted - they're economic/grid decisions
- **Forced outages** (U1, U2, U3, MO, SF) represent true unplanned downtime - this is what we want to prevent
- Using PI-tag stops alone would miss critical GADS metadata (cause codes, event classification)

## Pipeline Steps:
1. **Load GADS Forced Outages** — Fetch forced outage events from `gold.fact_gads_event`, excluding PO/RS and deratings (D1-D4, D)
2. **Create Event Timeline** — Extract start/end timestamps and asset mappings
3. **Generate Lookback Windows** — For each historical timestamp, check if forced event occurs in next 4h/8h/24h
4. **Build 15-Min Label Grid** — Create modeling dataset: `[timestamp, asset_id, hours_to_next_stop, event_type, cause]`
5. **Save to Gold Schema** — Write `ml.labels` for Phase 2 feature engineering

**GADS Event Types (Included - True Forced Outages Only):**
- **U1** = Unplanned Outage - Immediate (forced trip)
- **U2** = Unplanned Outage - Delayed (deteriorating condition)
- **U3** = Unplanned Outage - Postponed (planned but forced earlier)
- **MO** = Maintenance Outage (unscheduled maintenance)
- **SF** = Startup Failure (failed to start)

**Excluded Event Types:**
- **PO** = Planned Outage (scheduled maintenance)
- **RS** = Reserve Shutdown (economic/grid decision)
- **D1, D2, D3, D4, D** = Deratings (partial capacity loss — different sensor signature than full trips)

## Step 1: Load GADS Forced Outages

Retrieve all forced/unscheduled downtime events (everything except PO and RS) with timestamps, asset mappings, and root cause text.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta

# Load GADS forced stops (full outages only)
stop_events = spark.sql("""
    SELECT 
        event_uid,
        asset_id,
        REAL_START_DT as event_start,
        REAL_END_DT as event_end,
        DURATION_HRS as duration_hours,
        EVENT_TYPE_CD as event_type_code,
        EVENT_TYPE_DESC as event_type,
        EVENT_CAUSE_CD as cause_code,
        CAUSE_CODE_DESC as cause_desc,
        CAUSE_OF_EVENT as cause_text,
        EQUIPMENT_DESC as equipment_desc,
        EVENT_YEAR,
        EVENT_MONTH
    FROM gold.fact_gads_event
    WHERE EVENT_TYPE_CD IN ('U1', 'U2', 'U3', 'MO', 'SF')
      AND asset_id IS NOT NULL
      AND REAL_START_DT IS NOT NULL
    ORDER BY asset_id, event_start
""")

# Load GADS deratings (capacity loss only)
derate_events = spark.sql("""
    SELECT 
        event_uid,
        asset_id,
        REAL_START_DT as event_start,
        REAL_END_DT as event_end,
        DURATION_HRS as duration_hours,
        EVENT_TYPE_CD as event_type_code,
        EVENT_TYPE_DESC as event_type,
        EVENT_CAUSE_CD as cause_code,
        CAUSE_CODE_DESC as cause_desc,
        CAUSE_OF_EVENT as cause_text,
        EQUIPMENT_DESC as equipment_desc,
        EVENT_YEAR,
        EVENT_MONTH
    FROM gold.fact_gads_event
    WHERE EVENT_TYPE_CD IN ('D1', 'D3')
      AND asset_id IS NOT NULL
      AND REAL_START_DT IS NOT NULL
    ORDER BY asset_id, event_start
""")

# Keep union for timeline generation and outage exclusion
forced_events = stop_events.unionByName(derate_events)

print(f"✓ Loaded {stop_events.count():,} forced stop events")
print(f"✓ Loaded {derate_events.count():,} derate events")
print(f"✓ Loaded {forced_events.count():,} total forced events for timeline coverage")

print(f"\nForced stop event type distribution:")
stop_events.groupBy("event_type_code", "event_type").count().orderBy("event_type_code").show(truncate=False)

print(f"\nDerate event type distribution:")
derate_events.groupBy("event_type_code", "event_type").count().orderBy("event_type_code").show(truncate=False)

print(f"\nAsset coverage across all forced events:")
forced_events.groupBy("asset_id").count().orderBy(F.desc("count")).show(20, truncate=False)

print(f"\nSample stop events:")
stop_events.select("asset_id", "event_start", "duration_hours", "event_type", "cause_desc").show(5, truncate=False)

print(f"\nSample derate events:")
derate_events.select("asset_id", "event_start", "duration_hours", "event_type", "cause_desc").show(5, truncate=False)

## Step 2: Define Prediction Horizons and Time Windows

Set up lookback windows for label generation (4h, 8h, 24h before each unplanned event).

In [ ]:
# Define prediction horizons (how far ahead to predict)
HORIZONS = {
    '4h': 4,
    '8h': 8,
    '24h': 24
}

# Add lookback window columns for each horizon
# For each event, calculate the timestamp window where labels should be set
def add_prediction_windows(events_df):
    events_with_windows = events_df

    for horizon_name, hours in HORIZONS.items():
        events_with_windows = events_with_windows.withColumn(
            f"window_start_{horizon_name}",
            F.expr(f"event_start - INTERVAL {hours} HOURS")
        ).withColumn(
            f"window_end_{horizon_name}",
            F.col("event_start")
        )

    return events_with_windows

stop_events_with_windows = add_prediction_windows(stop_events)
derate_events_with_windows = add_prediction_windows(derate_events)

print(f"✓ Added prediction windows for horizons: {list(HORIZONS.keys())}")
print(f"\nSample stop event with windows:")
stop_events_with_windows.select(
    "asset_id",
    "event_start",
    "window_start_4h",
    "window_start_8h",
    "window_start_24h",
    "event_type",
    "cause_desc"
).show(3, truncate=False)

print(f"\nSample derate event with windows:")
derate_events_with_windows.select(
    "asset_id",
    "event_start",
    "window_start_4h",
    "window_start_8h",
    "window_start_24h",
    "event_type",
    "cause_desc"
).show(3, truncate=False)

## Step 3: Generate Historical Timeline Grid

Create 15-minute timestamp grid covering the historical period where we have sensor data.

In [ ]:
# Get date range from PI data (or specify manually)
forced_assets = forced_events.select("asset_id").distinct()
forced_assets.createOrReplaceTempView("forced_assets")

date_range = spark.table("gold.fact_pi").join(
    forced_assets,
    on="asset_id",
    how="inner"
).agg(
    F.min("Timestamp").alias("min_date"),
    F.max("Timestamp").alias("max_date")
).collect()[0]

start_date = date_range['min_date']
end_date = date_range['max_date']

print(f"Historical data range: {start_date} to {end_date}")

# Generate 15-minute timestamp grid
timeline = spark.sql(f"""
    SELECT 
        timestamp_bin,
        asset_id
    FROM (
        SELECT explode(sequence(
            to_timestamp('{start_date}'),
            to_timestamp('{end_date}'),
            INTERVAL 15 MINUTES
        )) as timestamp_bin
    )
    CROSS JOIN (
        SELECT DISTINCT asset_id 
        FROM forced_assets
    )
""")

print(f"\n✓ Generated timeline grid:")
print(f"  Total timestamps: {timeline.select('timestamp_bin').distinct().count():,}")
print(f"  Assets: {timeline.select('asset_id').distinct().count()}")
print(f"  Total rows: {timeline.count():,}")

timeline.orderBy("asset_id", "timestamp_bin").show(10)

## Step 4: Join Timeline with Event Windows

For each timestamp, determine if a forced downtime event occurs within the next 4h/8h/24h.

In [ ]:
# For each horizon, join timeline with stop and derate events to create labels
# A timestamp gets label=1 if an event starts within the horizon window

def add_labels_for_event_set(base_df, events_df, label_prefix):
    labeled = base_df

    for horizon_name, hours in HORIZONS.items():
        horizon_events = events_df.select(
            "asset_id",
            "event_start",
            "event_type",
            "cause_desc",
            f"window_start_{horizon_name}"
        ).withColumnRenamed("event_start", f"next_event_{label_prefix}_{horizon_name}") \
         .withColumnRenamed("event_type", f"event_type_{label_prefix}_{horizon_name}") \
         .withColumnRenamed("cause_desc", f"cause_{label_prefix}_{horizon_name}")

        labeled = labeled.join(
            horizon_events,
            (labeled.asset_id == horizon_events.asset_id) &
            (labeled.timestamp_bin >= horizon_events[f"window_start_{horizon_name}"]) &
            (labeled.timestamp_bin < horizon_events[f"next_event_{label_prefix}_{horizon_name}"]),
            "left"
        ).select(
            labeled["*"],
            horizon_events[f"next_event_{label_prefix}_{horizon_name}"],
            horizon_events[f"event_type_{label_prefix}_{horizon_name}"],
            horizon_events[f"cause_{label_prefix}_{horizon_name}"]
        )

        labeled = labeled.withColumn(
            f"hours_to_event_{label_prefix}_{horizon_name}",
            F.when(
                F.col(f"next_event_{label_prefix}_{horizon_name}").isNotNull(),
                (F.unix_timestamp(f"next_event_{label_prefix}_{horizon_name}") - F.unix_timestamp("timestamp_bin")) / 3600
            ).otherwise(None)
        )

        labeled = labeled.withColumn(
            f"label_{label_prefix}_{horizon_name}",
            F.when(F.col(f"next_event_{label_prefix}_{horizon_name}").isNotNull(), 1).otherwise(0)
        )

    return labeled

labels = timeline
labels = add_labels_for_event_set(labels, stop_events_with_windows, "stop")
labels = add_labels_for_event_set(labels, derate_events_with_windows, "derate")

print(f"✓ Labels generated for all stop and derate horizons")
print(f"\nStop label distribution:")
for horizon_name in HORIZONS.keys():
    dist = labels.groupBy(f"label_stop_{horizon_name}").count().orderBy(f"label_stop_{horizon_name}").collect()
    print(f"\nStop {horizon_name}:")
    for row in dist:
        print(f"  {row[0]}: {row[1]:,}")

print(f"\nDerate label distribution:")
for horizon_name in HORIZONS.keys():
    dist = labels.groupBy(f"label_derate_{horizon_name}").count().orderBy(f"label_derate_{horizon_name}").collect()
    print(f"\nDerate {horizon_name}:")
    for row in dist:
        print(f"  {row[0]}: {row[1]:,}")

## Step 5: Aggregate Labels Per Timestamp

Consolidate multiple events per timestamp (keep nearest event per horizon).

In [ ]:
# If multiple events exist within a horizon for the same timestamp, keep the nearest one
window_spec_stop_4h = Window.partitionBy("timestamp_bin", "asset_id").orderBy(F.col("hours_to_event_stop_4h").asc_nulls_last())
window_spec_stop_8h = Window.partitionBy("timestamp_bin", "asset_id").orderBy(F.col("hours_to_event_stop_8h").asc_nulls_last())
window_spec_stop_24h = Window.partitionBy("timestamp_bin", "asset_id").orderBy(F.col("hours_to_event_stop_24h").asc_nulls_last())
window_spec_derate_4h = Window.partitionBy("timestamp_bin", "asset_id").orderBy(F.col("hours_to_event_derate_4h").asc_nulls_last())
window_spec_derate_8h = Window.partitionBy("timestamp_bin", "asset_id").orderBy(F.col("hours_to_event_derate_8h").asc_nulls_last())
window_spec_derate_24h = Window.partitionBy("timestamp_bin", "asset_id").orderBy(F.col("hours_to_event_derate_24h").asc_nulls_last())

labels_dedupe = labels \
    .withColumn("rank_stop_4h", F.row_number().over(window_spec_stop_4h)) \
    .withColumn("rank_stop_8h", F.row_number().over(window_spec_stop_8h)) \
    .withColumn("rank_stop_24h", F.row_number().over(window_spec_stop_24h)) \
    .withColumn("rank_derate_4h", F.row_number().over(window_spec_derate_4h)) \
    .withColumn("rank_derate_8h", F.row_number().over(window_spec_derate_8h)) \
    .withColumn("rank_derate_24h", F.row_number().over(window_spec_derate_24h)) \
    .filter(
        (F.col("rank_stop_4h") == 1) &
        (F.col("rank_stop_8h") == 1) &
        (F.col("rank_stop_24h") == 1) &
        (F.col("rank_derate_4h") == 1) &
        (F.col("rank_derate_8h") == 1) &
        (F.col("rank_derate_24h") == 1)
    )

# Final aggregation to one row per (timestamp_bin, asset_id)
label_grid = labels_dedupe.groupBy("timestamp_bin", "asset_id").agg(
    F.max("label_stop_4h").alias("label_stop_4h"),
    F.max("label_stop_8h").alias("label_stop_8h"),
    F.max("label_stop_24h").alias("label_stop_24h"),
    F.min("hours_to_event_stop_4h").alias("hours_to_next_stop_4h"),
    F.min("hours_to_event_stop_8h").alias("hours_to_next_stop_8h"),
    F.min("hours_to_event_stop_24h").alias("hours_to_next_stop_24h"),
    F.max("label_derate_4h").alias("label_derate_4h"),
    F.max("label_derate_8h").alias("label_derate_8h"),
    F.max("label_derate_24h").alias("label_derate_24h"),
    F.min("hours_to_event_derate_4h").alias("hours_to_next_derate_4h"),
    F.min("hours_to_event_derate_8h").alias("hours_to_next_derate_8h"),
    F.min("hours_to_event_derate_24h").alias("hours_to_next_derate_24h"),
    F.coalesce(
        F.first("next_event_stop_4h", ignorenulls=True),
        F.first("next_event_derate_4h", ignorenulls=True)
    ).alias("next_event_4h"),
    F.coalesce(
        F.first("next_event_stop_8h", ignorenulls=True),
        F.first("next_event_derate_8h", ignorenulls=True)
    ).alias("next_event_8h"),
    F.coalesce(
        F.first("next_event_stop_24h", ignorenulls=True),
        F.first("next_event_derate_24h", ignorenulls=True)
    ).alias("next_event_24h"),
    F.coalesce(
        F.first("event_type_stop_4h", ignorenulls=True),
        F.first("event_type_derate_4h", ignorenulls=True)
    ).alias("event_type_4h"),
    F.coalesce(
        F.first("cause_stop_4h", ignorenulls=True),
        F.first("cause_derate_4h", ignorenulls=True)
    ).alias("cause_4h")
)

print(f"✓ Label grid created: {label_grid.count():,} rows")
print(f"\nFinal label distribution:")
for horizon in ['4h', '8h', '24h']:
    total = label_grid.count()
    stop_positive = label_grid.filter(F.col(f"label_stop_{horizon}") == 1).count()
    derate_positive = label_grid.filter(F.col(f"label_derate_{horizon}") == 1).count()
    stop_pct = (stop_positive / total * 100) if total > 0 else 0
    derate_pct = (derate_positive / total * 100) if total > 0 else 0
    print(f"  Stop {horizon}: {stop_positive:,} / {total:,} ({stop_pct:.2f}% positive)")
    print(f"  Derate {horizon}: {derate_positive:,} / {total:,} ({derate_pct:.2f}% positive)")

print(f"\nSample labels:")
label_grid.orderBy("asset_id", "timestamp_bin").show(10, truncate=False)

## Step 5b: Exclude Timestamps During Active Outages

Timestamps that fall *during* an active outage (event_start ≤ t ≤ event_end) have abnormal sensor readings but would get label=0 ("no stop coming"). This teaches the model the wrong pattern. We NULL out labels for these rows so they are excluded from training via `dropna`.

In [ ]:
# Exclude timestamps during active outages
# During an outage, sensor readings are abnormal but label=0 would teach wrong patterns
outage_windows = forced_events.select(
    F.col("asset_id").alias("outage_asset_id"),
    F.col("event_start").alias("outage_start"),
    F.col("event_end").alias("outage_end")
).filter(F.col("outage_end").isNotNull())

label_grid = label_grid.join(
    outage_windows,
    (label_grid.asset_id == outage_windows.outage_asset_id) &
    (label_grid.timestamp_bin >= outage_windows.outage_start) &
    (label_grid.timestamp_bin <= outage_windows.outage_end),
    "left"
).withColumn(
    "is_during_outage",
    F.when(F.col("outage_start").isNotNull(), 1).otherwise(0)
)

# Null out labels during outages so they are excluded from training
for label_prefix in ['stop', 'derate']:
    for h in ['4h', '8h', '24h']:
        label_grid = label_grid.withColumn(
            f"label_{label_prefix}_{h}",
            F.when(F.col("is_during_outage") == 1, None).otherwise(F.col(f"label_{label_prefix}_{h}"))
        )

# Drop join artifacts and deduplicate
label_grid = label_grid.drop("outage_asset_id", "outage_start", "outage_end") \
    .dropDuplicates(["timestamp_bin", "asset_id"])

during_outage_count = label_grid.filter(F.col("is_during_outage") == 1).count()
print(f"Marked {during_outage_count} timestamps as during-outage (labels set to NULL, excluded from training)")

## Step 6: Add Metadata and Clean Up

Add useful metadata columns for downstream analysis.

In [ ]:
# Add metadata columns
label_grid_final = label_grid \
    .withColumn("hour_of_day", F.hour("timestamp_bin")) \
    .withColumn("day_of_week", F.dayofweek("timestamp_bin")) \
    .withColumn("date_key", 
        F.expr("CAST(date_format(timestamp_bin, 'yyyyMMdd') AS INT)")) \
    .withColumn("label_source", F.lit("GADS_forced_downtime")) \
    .withColumn("generated_at", F.current_timestamp())

# Calculate overall hours to nearest stop (minimum across all stop horizons)
label_grid_final = label_grid_final.withColumn(
    "hours_to_next_stop",
    F.least(
        F.coalesce("hours_to_next_stop_4h", F.lit(999999)),
        F.coalesce("hours_to_next_stop_8h", F.lit(999999)),
        F.coalesce("hours_to_next_stop_24h", F.lit(999999))
    )
).withColumn(
    "hours_to_next_stop",
    F.when(F.col("hours_to_next_stop") == 999999, None).otherwise(F.col("hours_to_next_stop"))
)

# Calculate overall hours to nearest derate (minimum across all derate horizons)
label_grid_final = label_grid_final.withColumn(
    "hours_to_next_derate",
    F.least(
        F.coalesce("hours_to_next_derate_4h", F.lit(999999)),
        F.coalesce("hours_to_next_derate_8h", F.lit(999999)),
        F.coalesce("hours_to_next_derate_24h", F.lit(999999))
    )
).withColumn(
    "hours_to_next_derate",
    F.when(F.col("hours_to_next_derate") == 999999, None).otherwise(F.col("hours_to_next_derate"))
)

print(f"✓ Metadata added")
print(f"\nFinal schema:")
label_grid_final.printSchema()

print(f"\nSample rows with metadata:")
label_grid_final.select(
    "timestamp_bin", "asset_id",
    "hours_to_next_stop",
    "hours_to_next_derate",
    "label_stop_4h", "label_stop_8h", "label_stop_24h",
    "label_derate_4h", "label_derate_8h", "label_derate_24h",
    "event_type_4h", "cause_4h",
    "label_source"
).show(10, truncate=True)

## Step 7: Save Labels to ML Schema

Write `ml.labels` for Phase 2 feature engineering.

In [ ]:
# Save to gold schema (overwrite old PI-tag-based schema with new GADS-based schema)
label_grid_final.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("ml.labels")

print("✅ Label grid saved to ml.labels")
print(f"Total rows: {spark.table('ml.labels').count():,}")

# Show summary statistics
print("\n=== Label Summary ===")
spark.sql("""
    SELECT 
        COUNT(*) as total_timestamps,
        COUNT(DISTINCT asset_id) as num_assets,
        SUM(label_stop_4h) as stops_4h,
        SUM(label_stop_8h) as stops_8h,
        SUM(label_stop_24h) as stops_24h,
        SUM(label_derate_4h) as derates_4h,
        SUM(label_derate_8h) as derates_8h,
        SUM(label_derate_24h) as derates_24h,
        ROUND(AVG(hours_to_next_stop), 2) as avg_hours_to_stop,
        ROUND(AVG(hours_to_next_derate), 2) as avg_hours_to_derate
    FROM ml.labels
""").show(truncate=False)

# Show cause code distribution for upcoming stops
print("\n=== Top Stop Causes (4h horizon) ===")
spark.sql("""
    SELECT 
        cause_4h,
        COUNT(*) as label_count,
        COUNT(DISTINCT asset_id) as affected_assets
    FROM ml.labels
    WHERE label_stop_4h = 1
      AND cause_4h IS NOT NULL
    GROUP BY cause_4h
    ORDER BY label_count DESC
    LIMIT 10
""").show(truncate=False)

print("\n=== Top Derate Labels (4h horizon) ===")
spark.sql("""
    SELECT 
        COUNT(*) as derate_label_count,
        COUNT(DISTINCT asset_id) as affected_assets
    FROM ml.labels
    WHERE label_derate_4h = 1
""").show(truncate=False)

print("\n✅ Phase 1 Complete: split GADS stop and derate labels generated")
print("   Stop labels use U1/U2/U3/MO/SF; derate labels use D1/D3")
print("   Next: Phase 2 will build features for both prediction tracks")

## Step 8: Encode Failure Causes for Model Use

Create categorical features from CAUSE_CODE_DESC to help the model learn failure patterns.

In [ ]:
# Create cause code lookup table for model feature engineering
# This maps CAUSE_CODE_DESC to high-level categories for easier modeling

# Get all unique cause descriptions
all_causes = forced_events.select("cause_desc").distinct().orderBy("cause_desc")

print("=== Unique Failure Cause Descriptions ===")
print(f"Total unique causes: {all_causes.count()}")
all_causes.show(50, truncate=False)

# Create cause category mapping (you can refine these categories based on domain knowledge)
cause_categories = spark.createDataFrame([
    # Boiler-related
    ("boiler", ["boiler", "tube", "drum", "furnace", "burner", "sootblower"]),
    # Turbine-related
    ("turbine", ["turbine", "rotor", "blade", "shaft", "bearing"]),
    # Electrical
    ("electrical", ["electrical", "generator", "exciter", "transformer", "voltage", "breaker", "relay", "power supply", "I/O", "control"]),
    # Feed/Water systems
    ("feedwater", ["feedwater", "condensate", "pump", "valve", "piping"]),
    # Instrumentation & Controls
    ("controls", ["instrumentation", "control", "sensor", "indication", "monitoring"]),
    # Mechanical
    ("mechanical", ["mechanical", "vibration", "coupling", "seal", "gasket"]),
    # Other/Unknown
    ("other", ["other", "miscellaneous", "unknown"])
], ["category", "keywords"])

# Explode keywords and create mapping function
category_map = cause_categories.select(
    F.col("category"),
    F.explode("keywords").alias("keyword")
)

print("\n=== Cause Category Mapping ===")
category_map.show(truncate=False)

# This mapping will be used in Phase 2 to create categorical features
# Save for reference
category_map.write.mode("overwrite").format("delta").saveAsTable("gold.failure_cause_categories")

print("\n✅ Cause category mapping saved to gold.failure_cause_categories")
print("   Phase 2 will use this to create categorical features from CAUSE_CODE_DESC")